# CareLoop Clinics — No-Show Guard

## Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend

### Task 4.1 — One validated call

Get a free key in Google AI Studio and install the library in Colab. Write a function analyse_note(note_text) that sends one note to the Gemini Flash model used in class, with a system instruction, a prompt built with the CRAFT parts (Context, Role, Action, Tone, Format), temperature=0, response_mime_type set to JSON, and a JSON response schema with four fields: root_cause (a plain string — do not make it an enum; the allowed values go in your prompt and are enforced afterwards by the check in Task 4.2), confidence (0–1), summary (at most 25 words), recommended_action. Set the thinking level to minimal as shown in Session 22.1, and give max_output_tokens plenty of room (1000 or more): on some models hidden reasoning tokens count against the limit and can cut the JSON off, which makes json.loads fail. Run it on note N001 and print the parsed dictionary. Shape of the call (follow your Session 22.1 notebook if any detail differs; the key is typed at run time):

In [1]:
import pandas as pd
import json
import time

from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

In [2]:
notes_df = pd.read_csv("followup_notes.csv")

print("Shape:",notes_df.shape)
print("Columns",notes_df.columns.to_list())
print(notes_df.head())

Shape: (30, 3)
Columns ['note_id', 'appointment_id', 'note_text']
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [3]:
#Check all first 10 notes
for _, row in notes_df.head(10).iterrows():
  print(row['note_id'], ":" ,row['note_text'])


N001 : Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
N002 : Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
N003 : Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
N004 : nan
N005 : Had high fever and body ache on the day, stayed in bed. Will visit once recovered.
N006 : Manager called an urgent meeting at the office, could not leave work. Asked for an evening slot next time.
N007 : Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N008 : City bus strike that day, no auto available near the colony. Distance is a big problem for him.
N009 : Phone was switched off, could not reach the patient after three attempts.
N010 : Had a shift change at the factory and could not take leave. Prefers Saturday appointments.


In [18]:
from getpass import getpass

API_KEY = getpass("API_KEY").strip()

client = genai.Client(api_key = API_KEY)

model_name =  "gemini-3.5-flash-lite"
#"gemini-3.5-flash-lite"
print("API Key configured Successfully")



API_KEY··········
API Key configured Successfully


In [5]:
schema = {
    "type":"object",
    "properties":{
        "root_cause":{"type":"string"},
        "confidence":{"type":"number"},
        "summary":{"type":"string"},
        "recommended_action":{"type":"string"}
    },
    "required":[
      "root_cause",
      "confidence",
      "summary",
      "recommended_action"

    ]
}

In [6]:
SYSTEM_TEXT = """
You are a patient follow-up analysis assistant.
Analyse patient call-centre notes accurately.
Return only valid JSON matching the supplied schema.
Do not invent information that is not present in the note.
"""

def build_prompt(note_text):

    prompt = f"""
Context:
A clinic wants to understand why patients miss appointments.

Role:
You are an expert patient follow-up analyst.

Action:
Read the patient note and identify the main reason for the missed appointment.

Choose exactly one root_cause from:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other

If the reason is unclear or ambiguous, choose other rather than guessing.

Also provide:
1. Confidence between 0 and 1.
2. A short summary of at most 25 words.
3. A practical recommended action for the clinic.

Tone:
Be neutral, factual and concise.

Format:
Return only JSON with exactly these four fields:
root_cause, confidence, summary, recommended_action.

Patient note:
{note_text}
"""
    return prompt

In [16]:
note_001 = notes_df.loc[
    notes_df['note_id'] == 'N001', 'note_text'
].iloc[0]

result = analyse_and_validate(note_001)

print(result)
print("API calls:", api_call_count)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient missed the appointment because they did not receive a reminder and subsequently forgot the scheduled visit.', 'recommended_action': "Verify the patient's contact information in the system and confirm their preferred method for receiving automated appointment reminders.", 'validation_status': 'Valid', 'raw_response': ''}
API calls: 1


## Task 4.2 — Validate and process the 10 notes

In [7]:
#Pydantic Validation
class FollowupInsights(BaseModel):
  root_cause :str =Field(min_length=3)
  confidence : float = Field(ge =0, le = 1 )
  summary : str = Field (min_length = 5)
  recommended_action : str = Field (min_length = 5)

In [8]:
allowed_root_causes = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [9]:

api_call_count = 0


def analyse_and_validate(note_text):
    global api_call_count

    raw_response = ""
    data = {}

    try:
      prompt = build_prompt(note_text)

      response = client.models.generate_content(
          model=model_name,
          contents=prompt,
          config=types.GenerateContentConfig(
              system_instruction=SYSTEM_TEXT,
              temperature=0,
              max_output_tokens=1000,
              response_mime_type="application/json",
              response_schema=schema
          )
      )

      api_call_count += 1

      data = json.loads(response.text)

      validated = FollowupInsights(**data)

      if validated.root_cause  not in allowed_root_causes:
        raise ValueError("Invalid RootCause")

      return {
          "root_cause":validated.root_cause,
          "confidence":validated.confidence,
          "summary":validated.summary,
          "recommended_action":validated.recommended_action,
          "validation_status":"Valid",
          "raw_response": raw_response
      }

    except Exception as e:

          print("Validation/API error:", repr(e))
          print("Raw response:", raw_response)

          return {
              "root_cause": data.get("root_cause", ""),
              "confidence": data.get("confidence", ""),
              "summary": data.get("summary", ""),
              "recommended_action": data.get("recommended_action", ""),
              "validation_status": "invalid",
              "raw_response": raw_response
          }




In [11]:
#Step 2 — Process the first 10 notes

This code will:
- Skip N004 because its note is blank.
- Send the other 9 notes to Gemini.
- Validate each response using FollowupInsight.
- Wait 13 seconds between API calls to avoid rate limits.
- Store all results in a list.

In [23]:
results = []

first_10_notes = notes_df.head(10)

for _,row in first_10_notes.iterrows():
  notes_text = row['note_text']

  #Check whether the note is blank
  if pd.isna(notes_text) or str(notes_text).strip() == "":
    result = {
         "root_cause":"other",
          "confidence":0.0,
          "summary":"No note recorded",
          "recommended_action":"Re-call patient",
          "validation_status":"skipped_blank",
          "raw_response": ""

    }
  else:
    result = analyse_and_validate(notes_text)

    if result["validation_status"] != "Valid":
            print("Processing stopped due to API or validation error.")
            break

    #wait before next API call
    time.sleep(13)

  results.append({
     "note_id" : row["note_id"],
     "appointment_id" : row["appointment_id"],
     **result
 })

print("Total Notes Processed",len(results))
print("API Calls:",api_call_count)



Total Notes Processed 10
API Calls: 13


In [28]:
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

results_df.to_csv("notes_analyzed.csv",index = False)

print("Saved Successfully")
print("Shape", results_df.shape)
print("columns", results_df.columns.to_list())

Saved Successfully
Shape (10, 8)
columns ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']


## Task 4.3 — Break it on purpose

(a) Feed the Pydantic model two deliberately bad replies and catch each with try/except ValidationError (from pydantic import ValidationError): one with confidence = 1.7, one missing recommended_action; print what each error says. (b) For notes N007 (Hinglish) and N009 (ambiguous), compare two prompts at temperature=0: V1 = just "Classify this note", V2 = your CRAFT prompt with the allowed values and the instruction to answer other rather than guess. Use the same schema and settings for both (root_cause as a plain string, JSON mode, temperature=0), so that the only difference is the prompt; then check each reply against the allowed list. Show the four results in a small table and say which prompt you would deploy. (c) Count and print the total number of API calls you used across Tasks 4.1–4.3 (about 14 on a first pass: 1 in 4.1, 9 in 4.2, 4 here; 12 if you reuse your 4.2 results for the V2 prompt).

In [30]:
#Step 1 — Test an invalid confidence value
try:
  FollowupInsights(
      root_cause ="forgot_or_no_reminder",
      confidence = 1.7,
      summary="Patient forgot the appointment.",
      recommended_action="Send another reminder."
  )

except ValidationError as e:
  print("Validation Error")
  print(e)

Validation Error
1 validation error for FollowupInsights
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [32]:
#Step 2 : Test 2 — Missing field
try:
    FollowupInsights(
        root_cause="forgot_or_no_reminder",
        confidence=0.9,
        summary="Patient forgot the appointment."
    )

except ValidationError as e:
    print("Validation Error:")
    print(e)

Validation Error:
1 validation error for FollowupInsights
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or...orgot the appointment.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [45]:
#Prompt Comparison
#Prompt V1
def analyse_v1(note_text):

    global api_call_count

    prompt = f"""Classify the appointment no show note

    note : {note_text}

    Return the result using the required JSON fields.
    """

    response = client.models.generate_content(
        model = model_name,
        contents = prompt,
        config = types.GenerateContentConfig(
            system_instruction = SYSTEM_TEXT,
            temperature = 0,
            max_output_tokens = 1000,
            response_mime_type = "application/json",
            response_schema = schema
        )
    )

    data = json.loads(response.text)

    FollowupInsights(**data)

    if data['root_cause'] in allowed_root_causes:
        print("V1 validation: Valid")
    else:
        print("V1 validation: Invalid category")

    return data

In [49]:
#Test V1 on N007 and N009

note_007 = notes_df.loc[notes_df['note_id'] == 'N007','note_text'].iloc[0]

note_009 = notes_df.loc[notes_df['note_id'] == 'N009','note_text'].iloc[0]

v1_007 = analyse_v1(note_007)
time.sleep(13)



print("V1 N007:", v1_007)



V1 validation: Invalid category
V1 N007: {'root_cause': 'Forgot the appointment and did not receive an SMS reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because they forgot and claimed they did not receive an SMS reminder. They want to reschedule.', 'recommended_action': 'Help the patient book a new appointment and verify their mobile number for SMS notifications.'}


In [48]:
v1_009 = analyse_v1(note_009)
print("V1 N009:", v1_009)

V1 validation: Invalid category
V1 N009: {'root_cause': 'Patient unreachable', 'confidence': 1.0, 'summary': 'The patient could not be reached after three attempts because their phone was switched off.', 'recommended_action': 'Attempt to contact the patient again at a later time or send an alternate form of communication such as an email or letter.'}


In [50]:
#let's compare V1 with V2

for note_id in ["N007", "N009"]:
    row = results_df[results_df["note_id"] == note_id].iloc[0]

    print("\nNote:", note_id)
    print("V2 Root Cause:", row["root_cause"])
    print("V2 Validation:", row["validation_status"])


Note: N007
V2 Root Cause: forgot_or_no_reminder
V2 Validation: Valid

Note: N009
V2 Root Cause: other
V2 Validation: Valid


The simple V1 prompt produced meaningful descriptions but failed the allowed root-cause validation for both N007 and N009. The CRAFT-based V2 prompt returned valid categories for both notes. This demonstrates that structured instructions and clearly defined output constraints improve the consistency of GenAI classification.

In [51]:
print("Total API calls:", api_call_count)

Total API calls: 13


## Task 4.4 — Recommendation and GitHub

(a) (3 marks) Merge notes_analysed.csv with appointments_clean.csv on appointment_id and cross-tabulate root_cause against reminder_sent (pd.crosstab). Write README.md with: a 4-line summary (one finding per Part, each with its number); a one-paragraph recommendation that answers the business question using at least four numbers from Parts 1–4; the model you chose and why (include AUC and the threshold); and three limitations (choose from: the 10 notes are an illustration and cannot show the real reasons clinic-wide; the cleaning statistics came from all rows before the split; the same patient can sit in both train and test; the split is random, not past-to-future; one train/test split; a threshold that really depends on call capacity; clinic-level sample sizes; the data are observational, so associations are not causes). (b) (2 marks) Cre

In [56]:
#Load Appointmens Clean csv and check
appointments_df = pd.read_csv("appointments_clean.csv")
print(appointments_df.head(5))
print("Shape:", appointments_df.shape)
print("columns:", appointments_df.columns.to_list())

  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  
1         

In [57]:
#Merge the analysed notes with appointments
merged_df = results_df.merge(appointments_df,
                             on ="appointment_id",
                             how ="left")
print("Merged Shape:", merged_df.shape)
print("Columns:", merged_df.columns.tolist())

Merged Shape: (10, 23)
Columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [58]:
cause_reminder_table = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print(cause_reminder_table)

reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1


This analysis helps the clinic understand that not every missed appointment has the same reason.
- Forgotten appointments may need better reminder systems.
- Transport problems may need travel assistance or easier rescheduling.
- Cost-related problems may need follow-up discounts or flexible payment options.
- Work-related problems may need evening or weekend slots.
- Illness-related problems may need easy rescheduling.

The root-cause analysis was merged with the appointment dataset using appointment_id, producing 10 records and 23 columns. The crosstab showed that both patients classified under forgot_or_no_reminder had no reminder sent. Overall, five appointments had no reminder and five had a reminder. Other reasons included transport, cost, illness and work-related issues. These findings suggest that clinics should consider different follow-up strategies based on the reason for missed appointments.